# Flash Sale: NoSQL, Reliability & Scalability
**P1W3D5 · 60 menit · VS Code · Berpasangan**

Nama: **isi di sini**

Anda adalah engineer marketplace. Pilih model data dan amati apa yang terjadi ketika
jaringan antar replika terputus dan traffic terkonsentrasi pada produk tertentu.

Buka juga `02_Lembar_Jawaban.md`. Prediksi sebelum menjalankan cell, kemudian catat hasil.
Semua kode awal dapat langsung dijalankan. Tugas Anda mengubah parameter yang ditandai
**TUGAS**, melakukan eksperimen, dan menjelaskan hasil. Setup selesai sebelum kelas.

| Menit | Aktivitas |
|---|---|
| 00–05 | Briefing dan pemeriksaan kesiapan |
| 05–15 | Model data |
| 15–30 | Replication dan partition |
| 30–45 | Sharding |
| 45–55 | Quorum |
| 55–60 | Debrief dan exit ticket |

Node adalah dictionary di satu proses. Ini simulasi perilaku dengan asumsi eksplisit,
bukan cluster database sungguhan atau pengujian performa.

## 0. Persiapan (00–05)
Pilih kernel Python `.venv`. Jalankan cell ini. Jika import gagal, pastikan notebook berada
di folder yang sama dengan `simulator.py` dan folder `data` ikut diekstrak.

In [ ]:
from copy import deepcopy
from pprint import pprint
import sys
from simulator import (
    load_products, load_traffic, new_cluster, leader_write, replica_read,
    sync_from_primary, range_shard, hash_shard, shard_load, show_load,
    quorum_write, quorum_read,
)
products = load_products()
traffic = load_traffic('traffic_flash_sale.csv')
print('Python:', sys.version.split()[0])
print(f'Siap: {len(products)} produk | {len(traffic)} request flash sale')

## 1. Pilihan model data (05–15)
**3 menit:** periksa dua dokumen berikut. Bandingkan atribut elektronik dan fashion.
Tambahkan satu atribut relevan pada salinan dokumen, lalu cetak hasilnya.

**7 menit:** isi enam pilihan model data di lembar jawaban. Untuk setiap fitur,
jelaskan pola query/penulisan yang membuat pilihan itu masuk akal. Gunakan relational
untuk kebutuhan yang sesuai. Banyak sistem memakai lebih dari satu jenis penyimpanan.

Kita tidak menjalankan enam jenis database. Dictionary di cell hanya ilustrasi dokumen JSON.

In [ ]:
pprint(products[:2])
my_product = deepcopy(products[0])
# TUGAS: ganti nama atribut dan nilainya dengan atribut relevan pilihan Anda.
attribute_name = 'warranty_months'
attribute_value = 12
my_product['attributes'][attribute_name] = attribute_value
print('Dokumen setelah perubahan:')
pprint(my_product)

## 2. Replication dan network partition (15–30)
**3 menit:** A adalah primary. A dan B dapat berkomunikasi, C terisolasi. Stok awal 10.
Write stok 7 mencapai A-B dan diterima oleh mayoritas. Prediksi apa yang akan dibaca C.

**7 menit:** jalankan kedua kebijakan. `serve_local` membaca salinan lokal, sedangkan
`require_majority` dalam model ini membutuhkan akses ke primary A dan minimal dua node.
Ubah stok baru menjadi 5 dan ulangi cell. Catat hasil percobaan awal 7 dahulu.

**3 menit:** pulihkan komunikasi dan jalankan sinkronisasi manual.
**2 menit:** isi hasil dan trade-off di lembar jawaban.

Availability dalam CAP berarti operasi mendapat respons yang memenuhi spesifikasinya.
Penolakan merupakan kegagalan melayani operasi tersebut, meski server mengirim respons error.

In [ ]:
# Cell aman diulang: selalu mulai dari cluster baru.
cluster = new_cluster()
new_stock = 7  # TUGAS: setelah mencatat hasil awal, ubah menjadi 5 dan ulangi.
print('Awal:', cluster)
print('Write di sisi A-B:', leader_write(cluster, new_stock, reachable=['A', 'B']))
print('Setelah write:', cluster)
print('C melayani lokal:', replica_read(cluster, 'C', 'serve_local', ['C']))
print('C membutuhkan mayoritas:', replica_read(cluster, 'C', 'require_majority', ['C']))

In [ ]:
# Pulihkan jaringan dan salin versi primary ke semua replika.
# Jalankan cell eksperimen sebelumnya lagi jika ingin mengulang keadaan partition.
print('Sesudah sinkronisasi manual:', sync_from_primary(cluster))
print('C membaca lagi:', replica_read(cluster, 'C', 'serve_local', ['A', 'B', 'C']))

**Diskusi:** kapan stale read masih dapat diterima? Apa risikonya untuk stok checkout?
Salinan data ini tidak otomatis mencegah overselling tanpa mekanisme transaksi/konkurensi.
Lab belum mensimulasikan concurrent checkout, leader election, atau failover otomatis.

## 3. Sharding dan hotspot (30–45)
Tukar peran dengan pasangan.

**3 menit:** pelajari pembagian ID. Range: 1–10, 11–20, 21–30. Hash: `product_id % 3`.
Kedua strategi menyimpan 10 produk per shard untuk dataset ini.

**5 menit:** jalankan traffic flash sale, lalu ganti `traffic_file` ke
`traffic_hot_product.csv` dan ulangi. Isi empat baris distribusi pada lembar jawaban.
Output menghitung **request**, bukan jumlah produk atau waktu respons.

**4 menit:** telusuri produk ID 30. Mengapa seluruh request untuk key itu menuju satu shard?
**3 menit:** tulis mitigasi dan bedakan replication dari sharding.

Hash di sini adalah modulo sederhana, bukan consistent hashing. Saat jumlah shard diubah,
banyak key dapat berpindah. Kita belum mengukur migrasi atau overhead koordinasi.

In [ ]:
product_ids = [row['product_id'] for row in products]
print('Jumlah produk per shard:')
print('Range:', shard_load(product_ids, range_shard))
print('Hash :', shard_load(product_ids, hash_shard))

In [ ]:
# TUGAS: setelah mencatat flash sale, ubah ke 'traffic_hot_product.csv'.
traffic_file = 'traffic_flash_sale.csv'
requests = load_traffic(traffic_file)
print('Dataset:', traffic_file, '| request:', len(requests))
show_load('Range', shard_load(requests, range_shard))
show_load('Hash', shard_load(requests, hash_shard))
print('Produk 30 berada di shard range:', range_shard(30), 'dan hash:', hash_shard(30))

## 4. Tunable consistency / quorum (45–55)
**2 menit:** N=3. W adalah minimum replika yang mengonfirmasi write. R adalah minimum
respons replika untuk read. Jika node yang dapat dijangkau kurang dari ambang, operasi ditolak.

**5 menit:** jalankan empat kasus di bawah, lalu isi tabel jawaban. Tiap kasus mulai dari
stok 10. Write 7 hanya mencapai A-B. Respons read dipilih dari daftar `read_nodes` secara
berurutan, sebanyak R. Read memilih versi tertinggi di antara respons yang diterima.

**2 menit:** pada cell eksperimen bebas, ubah write agar hanya mencapai C. Bandingkan W=1
lalu W=2. Prediksi dulu. Tentukan sendiri R dan node baca, lalu catat hasilnya.
**1 menit:** jelaskan efek menaikkan W/R saat jaringan bermasalah.

**Batas penting:** W+R>N menghasilkan irisan himpunan write/read pada replika tetap.
Dalam lab, satu penulis menghasilkan versi berurutan, write selesai sebelum read, dan read
memilih versi tertinggi. Irisan saja belum menjamin linearizability pada sistem nyata dengan
concurrent write, sloppy quorum, perubahan anggota, dan resolusi konflik.

In [ ]:
cases = [
    (1, 1, ['C']),
    (2, 2, ['C']),
    (2, 2, ['B', 'C']),
    (3, 1, ['A', 'B']),
]
for W, R, read_nodes in cases:
    q = new_cluster()
    written = quorum_write(q, stock=7, W=W, reachable=['A', 'B'])
    read = quorum_read(q, R=R, reachable=read_nodes)
    print(f'W={W}, R={R}, read_nodes={read_nodes}')
    print('  Write:', written)
    print('  Read :', read)

In [ ]:
# TUGAS: ubah W dari 1 menjadi 2 dan ulangi. Pilih juga R serta node baca.
W = 1
R = 1
write_nodes = ['C']
read_nodes = ['C']
q = new_cluster()
print('Write:', quorum_write(q, stock=7, W=W, reachable=write_nodes))
print('Read :', quorum_read(q, R=R, reachable=read_nodes))
print('Isi node:', q)

## 5. Debrief dan exit ticket (55–60)
Jawab tiga pertanyaan exit ticket pada lembar jawaban. Bahas satu temuan dengan pasangan.
Simpan notebook dengan output dan lembar jawaban. Jangan memasukkan `.venv` ke pengumpulan.

Sebelum menyerahkan, pastikan hasil kedua dataset sharding sudah dicatat di lembar jawaban,
meski cell notebook hanya menampilkan dataset terakhir. Hasil quorum yang ditolak bukan error
Python, melainkan perilaku sistem yang perlu dijelaskan.

### Referensi
- Materi kelas: P1W3D5, slide 3, 8, 11–16, 18–27, dan 31.
- [Jupyter Notebooks in VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks)
- [CAP Twelve Years Later](https://www.infoq.com/articles/cap-twelve-years-later-how-the-rules-have-changed/)
- [Dynamo paper, Amazon](https://www.amazon.science/publications/dynamo-amazons-highly-available-key-value-store)

Tidak semua NoSQL memakai eventual consistency. Pilihan model data dan jaminan transaksi
merupakan dimensi berbeda. Database relational juga dapat mendukung data fleksibel dan scaling.